# VidhanAI QLoRA Fine-Tuning (Phase 2) — Llama-3.2-3B

Kaggle-ready notebook that reproduces the paper's fine-tuning setup:

- **Base model:** `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` (matches the IEEE paper's Table II — the prior notebook trained Llama-3-8B, which does **not** match the paper).
- **Quantization:** 4-bit NF4
- **LoRA:** r=16, alpha=16, target modules q/k/v/o/gate/up/down
- **Training:** lr=2e-4, AdamW-8bit, batch 2, grad-accum 4 (effective 8), max_steps 60, max_seq_len 2048
- **Platform:** Kaggle T4 x2 (30h/week free GPU)

**Workflow:**
1. Create a Kaggle Dataset containing `train.jsonl` (and optionally `val.jsonl`) from `backend/scripts/ml_pipeline/datasets/`.
2. In Kaggle, click **Add Input** → your dataset, and set the path below (`/kaggle/input/<NAME>/train.jsonl`).
3. Run all cells. The adapter is saved to `lora_model/`.
4. Download `lora_model/` and place it at the repo's `notebooks/lora_model/`, then run local evaluation via `3_calculate_metrics.py`.

In [ ]:
# @title Install dependencies
# Unsloth pulls a recent `transformers` (>= 4.41) which renamed
# Trainer.__init__'s `tokenizer=` to `processing_class=`. To match that, we
# also need `trl` >= 0.10 (which exposes `processing_class=` on SFTTrainer).
# Pinning trl<0.9 (as older recipes do) breaks because SFTTrainer's
# super().__init__ then rejects `processing_class=` while transformers.Trainer
# simultaneously rejects `tokenizer=` — a version skew where neither kwarg
# works.
!pip install --upgrade --no-deps "trl>=0.11.0,<0.14.0" "transformers>=4.41.0,<4.50.0"
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.28" peft accelerate bitsandbytes
!pip install datasets evaluate rouge_score sacrebleu

In [ ]:
import torch, os, json
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
# @title Load base model in 4-bit (NF4)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",     # paper: 4-bit NF4
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
# Required for gradient checkpointing to work in 4-bit models.
model.config.use_cache = False

print("Loaded:", BASE_MODEL)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

In [ ]:
# @title Configure LoRA (r=16, alpha=16, q/k/v/o/gate/up/down)
# prepare_model_for_kbit_training handles:
#   - gradient checkpointing
#   - making output embedding layer & layernorm trainable for stable training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=RANK,
    lora_alpha=ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],  # paper: attention + MLP
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# @title Format prompts (chat template) and pre-tokenize
# We pre-tokenize the dataset here (returning input_ids + labels) instead of
# relying on `dataset_text_field="text"`. The reason: newer trl SFTTrainer
# applies an instruction-only completion mask to plain-text fields, which on
# Llama-3 chat templates can mask the assistant tokens entirely, producing
# loss = 0 (which then crashes the TrainingArguments.log() path with
# `AttributeError: '0' object has no attribute 'item'`).
# Returning explicit labels = input_ids forces the causal-LM loss to cover
# every token, which is exactly what we want for SFT.
EOS_TOKEN = tokenizer.eos_token

SYSTEM_PROMPT = (
    "You are an expert legal translator. Summarize Indian legislative text "
    "into plain, accessible English suitable for a high school reading level. "
    "Retain all factual penalties, dates, and jurisdictions. Format output clearly."
)

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    input_ids_list, attention_mask_list, labels_list = [], [], []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"{instruction}\n\n{input_text}"},
            {"role": "assistant", "content": output},
        ]
        full_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        ) + EOS_TOKEN
        result = tokenizer(
            full_text,
            truncation=True,
            max_length=max_seq_length,
            padding=False,
        )
        input_ids = result["input_ids"]
        attention_mask = result["attention_mask"]
        # Causal LM: labels = input_ids (no masking).
        labels = input_ids.copy()
        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)
    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list,
    }


train_ds = load_dataset("json", data_files=TRAIN_PATH, split="train")
train_ds = train_ds.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=train_ds.column_names,  # drop the original text columns
)
print("Train rows:", len(train_ds))

val_ds = None
if os.path.exists(VAL_PATH):
    val_ds = load_dataset("json", data_files=VAL_PATH, split="train")
    val_ds = val_ds.map(
        formatting_prompts_func,
        batched=True,
        remove_columns=val_ds.column_names,
    )
    print("Val rows:", len(val_ds))
else:
    print("No val.jsonl found; skipping eval split.")

In [ ]:
# @title Trainer (plain Trainer; bypasses unsloth's broken training step)
# Pre-tokenized dataset (input_ids + attention_mask + labels). Dynamic-padding
# collator: pads each batch to the longest sequence in *that* batch.
# DataCollatorForLanguageModeling with `mlm=False` chokes here because it
# expects pre-padded inputs; the custom collator below is simple and stable.
PAD_ID = tokenizer.pad_token_id

def collate(batch):
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids, attn, labels = [], [], []
    for b in batch:
        n = max_len - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [PAD_ID] * n)
        attn.append(b["attention_mask"] + [0] * n)
        labels.append(b["labels"] + [-100] * n)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        learning_rate=LR,
        fp16=USE_FP16,
        bf16=USE_BF16,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir="outputs",
        report_to="none",
        save_strategy="no",          # final save happens in cell-8
        eval_strategy="no",
        do_eval=False,
    ),
    train_dataset=train_ds,
    processing_class=tokenizer,
    data_collator=collate,
)

In [ ]:
# @title Train (~5-10 min on T4 for 60 steps, 3B model)
trainer_stats = trainer.train()
print(trainer_stats)

In [ ]:
# @title Export adapter
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Saved adapter to lora_model/")
print("Sizes:")
!ls -lh lora_model/

In [ ]:
# @title (Optional) Quick inference sanity check
from unsloth import FastLanguageModel

infer_model, infer_tok = FastLanguageModel.from_pretrained(
    model_name="lora_model",
    max_seq_length=2048,
    load_in_4bit=True,
)
infer_model = FastLanguageModel.for_inference(infer_model)

sample = """The Health Security and National Security Cess Bill, 2025 was introduced in Lok Sabha on December 1, 2025. The Bill proposes to levy a cess on production of goods such as pan masala."""
messages = [
    {"role": "system", "content": "You are an expert legal translator. Summarize Indian legislative text into plain accessible English, retaining all penalties, dates, and jurisdictions."},
    {"role": "user", "content": f"Simplify this legal text:\n\n{sample}"},
]
prompt = infer_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = infer_tok(prompt, return_tensors="pt").to("cuda")
outputs = infer_model.generate(**inputs, max_new_tokens=256, temperature=0.3)
print(infer_tok.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

## Next steps after training

1. **Download** the `lora_model/` directory from the Kaggle notebook session (File → Download, or use the Kaggle API output).
2. **Place it** at the repo path `notebooks/lora_model/` (overwriting the current 133-byte stub).
3. On your machine, set `VIDHANAI_USE_LORA=1` and run:
   ```bash
   cd backend/scripts/ml_pipeline
   python 3_calculate_metrics.py
   ```
   This runs the honest zero-shot-baseline vs LoRA-finetuned evaluation with bootstrap CIs and writes `docs/metrics_summary.json`.
4. Regenerate the hallucination audit (`docs/failure_cases.md`) against the real adapter.

### Uploading the dataset to Kaggle

```bash
# after generating datasets locally:
cd backend/scripts/ml_pipeline
kaggle datasets init -p datasets   # fills in metadata
# edit datasets/dataset-metadata.json, then:
kaggle datasets create -p datasets
```